# FlashEats — Class 7 Workflow Model (Local Solved)

Solved locally against the submission repository. This notebook builds the order-level workflow model and connects it to the late-delivery KPI.

In [1]:
from pathlib import Path
import json, sqlite3, sys, time, subprocess
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

def find_root():
    candidates = [Path.cwd(), *Path.cwd().parents, Path(r"C:\Users\piyus\Downloads\FDE\flasheats-assignment-submission")]
    for p in candidates:
        if (p / "data" / "raw" / "sqlite" / "flasheats.db").exists():
            return p
    raise FileNotFoundError("Could not find the submission repo. Open this notebook from the submission folder or update ROOT.")

ROOT = find_root()
DB = ROOT / "data" / "raw" / "sqlite" / "flasheats.db"
DATA = ROOT / "data" / "raw" / "files"
OUT = ROOT / "outputs" / "classroom_challenges"
OUT.mkdir(parents=True, exist_ok=True)
con = sqlite3.connect(DB)
orders = pd.read_sql("SELECT * FROM orders", con)
orders["created_at"] = pd.to_datetime(orders["created_at"], errors="coerce")
orders["promised_eta"] = pd.to_datetime(orders["promised_eta"], errors="coerce")
orders["pickup_at"] = pd.to_datetime(orders["pickup_at"], errors="coerce")
orders["actual_delivery_at"] = pd.to_datetime(orders["actual_delivery_at"], errors="coerce")
orders = orders.drop_duplicates("order_id", keep="first")
outcomes = pd.read_csv(DATA / "order_outcomes.csv")
events = pd.read_csv(DATA / "order_events.csv")
tickets = pd.read_csv(DATA / "support_tickets.csv")
actions = pd.read_csv(DATA / "customer_app_actions.csv")
interventions = pd.read_csv(DATA / "order_interventions.csv")
restaurant_status = pd.read_csv(DATA / "restaurant_status.csv")
driver_events = json.loads((DATA / "driver_events.json").read_text(encoding="utf-8"))


## Challenge 1 — Reconstruct three order lifecycles

In [2]:
out = outcomes.drop_duplicates("order_id"); late_id=out.loc[out["late_flag"].eq(1),"order_id"].iloc[0]; ontime_id=out.loc[out["late_flag"].eq(0),"order_id"].iloc[0]; intervention_id=interventions["order_id"].iloc[0]
def build_order_timeline(order_id):
    rows=[]
    o=orders[orders.order_id.eq(order_id)].iloc[0]
    for col, typ in [("created_at","order_created"),("promised_eta","promised_eta"),("pickup_at","pickup"),("actual_delivery_at","delivered")]:
        if pd.notna(o[col]): rows.append({"event_time":o[col],"event_type":typ,"actor":"system","source_system":"orders"})
    for _,e in events[events.order_id.eq(order_id)].iterrows(): rows.append({"event_time":e.event_time,"event_type":e.event_type,"actor":e.actor_type,"source_system":e.source_system})
    for _,i in interventions[interventions.order_id.eq(order_id)].iterrows(): rows.append({"event_time":i.intervention_at,"event_type":i.intervention_type,"actor":i.initiated_by,"source_system":"interventions"})
    return pd.DataFrame(rows).assign(event_time=lambda d:pd.to_datetime(d.event_time,errors="coerce")).sort_values("event_time")
for label, oid in [("on_time",ontime_id),("late",late_id),("intervention",intervention_id)]: print(label,oid); display(build_order_timeline(oid))

on_time O00003


,event_time,event_type,actor,source_system
0,2026-08-01 19:35:00.000000,order_created,system,orders
3,2026-08-01 19:35:00.000000,ORDER_CREATED,customer,order_platform
1,2026-08-01 19:57:50.820366,pickup,system,orders
4,2026-08-01 19:57:50.820366,PICKED_UP,driver,order_platform
2,2026-08-01 20:13:28.061191,delivered,system,orders
5,2026-08-01 20:13:28.061191,DELIVERED,driver,order_platform


late O00001


,event_time,event_type,actor,source_system
0,2026-08-13 12:56:00.000000,order_created,system,orders
4,2026-08-13 12:56:00.000000,ORDER_CREATED,customer,order_platform
7,2026-08-13 13:08:00.000000,DRIVER_REASSIGNMENT,dispatch,interventions
2,2026-08-13 13:35:49.825561,pickup,system,orders
5,2026-08-13 13:35:49.825561,PICKED_UP,driver,order_platform
1,2026-08-13 14:11:36.000000,promised_eta,system,orders
3,2026-08-13 14:22:25.035899,delivered,system,orders
6,2026-08-13 14:22:25.035899,DELIVERED,driver,order_platform


intervention O00781


,event_time,event_type,actor,source_system
0,2026-08-22 12:33:00.000000,order_created,system,orders
4,2026-08-22 12:33:00.000000,ORDER_CREATED,customer,order_platform
7,2026-08-22 12:57:00.000000,PRIORITY_DISPATCH,operations,interventions
2,2026-08-22 13:11:17.309762,pickup,system,orders
5,2026-08-22 13:11:17.309762,PICKED_UP,driver,order_platform
1,2026-08-22 13:48:36.000000,promised_eta,system,orders
3,2026-08-22 14:01:40.289785,delivered,system,orders
6,2026-08-22 14:01:40.289785,DELIVERED,driver,order_platform


## Challenge 2 — Canonical project model

In [3]:
model = pd.DataFrame([
 {"table":"orders","primary_key":"order_id","foreign_keys":"customer_id, restaurant_id, driver_id","grain":"one row per source order"},
 {"table":"customer_app_actions","primary_key":"action_id","foreign_keys":"order_id, customer_id","grain":"one customer action"},
 {"table":"support_tickets","primary_key":"ticket_id","foreign_keys":"order_id","grain":"one support ticket"},
 {"table":"order_interventions","primary_key":"intervention_id","foreign_keys":"order_id","grain":"one intervention"},
 {"table":"order_outcomes","primary_key":"order_id","foreign_keys":"order_id","grain":"one canonical outcome"},]); display(model)
print("This model keeps source grains explicit and aggregates one-to-many interactions before joining to one order row, preventing fan-out and double counting.")

,table,primary_key,foreign_keys,grain
0,orders,order_id,"customer_id, restaurant_id, driver_id",one row per source order
1,customer_app_actions,action_id,"order_id, customer_id",one customer action
2,support_tickets,ticket_id,order_id,one support ticket
3,order_interventions,intervention_id,order_id,one intervention
4,order_outcomes,order_id,order_id,one canonical outcome


This model keeps source grains explicit and aggregates one-to-many interactions before joining to one order row, preventing fan-out and double counting.


## Challenge 3 — Interaction → intervention → outcome order model

In [4]:
action_agg=actions.groupby("order_id").agg(support_opened=("action_id","size"),cancel_attempted=("action_type",lambda s:int(s.astype(str).str.contains("cancel",case=False).any()))).reset_index(); action_agg["support_opened"]=action_agg["support_opened"]>0
int_agg=interventions.groupby("order_id").agg(intervention_count=("intervention_id","size"),intervention_types=("intervention_type",lambda s:", ".join(sorted(set(s.astype(str)))))).reset_index()
journey=outcomes.merge(orders[["order_id","customer_id","final_status"]],on="order_id",how="left").merge(action_agg,on="order_id",how="left").merge(int_agg,on="order_id",how="left"); journey[["support_opened","cancel_attempted"]]=journey[["support_opened","cancel_attempted"]].fillna(False); journey["intervention_count"]=journey["intervention_count"].fillna(0).astype(int); journey["intervention_types"]=journey["intervention_types"].fillna(""); display(journey.head()); print({"late_with_support":int(journey.query("late_flag == 1 and support_opened == True").shape[0]),"orders_with_intervention":int((journey.intervention_count>0).sum()),"most_common_intervention":interventions["intervention_type"].value_counts().idxmax(),"late_no_intervention_with_support":int(journey.query("late_flag == 1 and support_opened == True and intervention_count == 0").shape[0])})

C:\Users\piyus\AppData\Local\Temp\ipykernel_2796\631195741.py:3: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  journey=outcomes.merge(orders[["order_id","customer_id","final_status"]],on="order_id",how="left").merge(action_agg,on="order_id",how="left").merge(int_agg,on="order_id",how="left"); journey[["support_opened","cancel_attempted"]]=journey[["support_opened","cancel_attempted"]].fillna(False); journey["intervention_count"]=journey["intervention_count"].fillna(0).astype(int); journey["intervention_types"]=journey["intervention_types"].fillna(""); display(journey.head()); print({"late_with_support":int(journey.query("late_flag == 1 and support_opened == True").shape[0]),"orders_with_intervention":int((journey.intervention_count>0).sum()),"most_common_intervention":i

,order_id,final_status_norm,delivered_flag,late_flag,delay_min,outcome_bucket,customer_id,final_status,support_opened,cancel_attempted,intervention_count,intervention_types
0,O00001,delivered,1,1.0,10.82,delivered_late,C0168,delivered,True,0.0,1,DRIVER_REASSIGNMENT
1,O00002,delivered,1,1.0,3.97,delivered_late,C0043,delivered,False,False,0,
2,O00003,delivered,1,0.0,-4.41,delivered_on_time,C0855,delivered,False,False,0,
3,O00004,cancelled,0,NaN,NaN,cancelled,C0229,cancelled,False,False,0,
4,O00005,delivered,1,1.0,21.02,delivered_late,C0040,delivered,False,False,0,


{'late_with_support': 563, 'orders_with_intervention': 430, 'most_common_intervention': 'DRIVER_REASSIGNMENT', 'late_no_intervention_with_support': 410}


## Challenge 4 — Selected business metrics

In [5]:
metrics = pd.DataFrame([
 {"metric":"Late delivery rate","formula":"late outcomes / measurable delivered outcomes","grain":"order","why":"project outcome KPI"},
 {"metric":"Customer friction rate","formula":"orders with customer action or ticket / orders","grain":"order","why":"customer interaction leading indicator"},
 {"metric":"Intervention rate","formula":"orders with ≥1 intervention / orders","grain":"order","why":"controllable operational response"},
 {"metric":"Median delay among late orders","formula":"median delay_min where late_flag=1","grain":"late order","why":"severity of outcome"},]); display(metrics)
print("Calculated values:"); print({"late_rate_pct":round(journey.late_flag.mean()*100,2),"customer_friction_rate_pct":round((journey.support_opened | journey.cancel_attempted).mean()*100,2),"intervention_rate_pct":round((journey.intervention_count>0).mean()*100,2),"median_delay_min":round(journey.loc[journey.late_flag.eq(1),"delay_min"].median(),2)})

,metric,formula,grain,why
0,Late delivery rate,late outcomes / measurable delivered outcomes,order,project outcome KPI
1,Customer friction rate,orders with customer action or ticket / orders,order,customer interaction leading indicator
2,Intervention rate,orders with ≥1 intervention / orders,order,controllable operational response
3,Median delay among late orders,median delay_min where late_flag=1,late order,severity of outcome


Calculated values:
{'late_rate_pct': np.float64(56.39), 'customer_friction_rate_pct': np.float64(68.75), 'intervention_rate_pct': np.float64(26.88), 'median_delay_min': np.float64(8.03)}


## Challenge 5 — Workflow investigations

In [6]:
journey["has_intervention"]=journey.intervention_count>0
print("A. Support association"); display(journey.groupby("support_opened").agg(orders=("order_id","size"),late_rate_pct=("late_flag",lambda x:round(x.mean()*100,2)),median_delay_min=("delay_min","median")))
print("B. Intervention association"); display(journey.groupby("has_intervention").agg(orders=("order_id","size"),late_rate_pct=("late_flag",lambda x:round(x.mean()*100,2))))
print("C. Intervention type"); display(journey[journey.has_intervention].explode("intervention_types").groupby("intervention_types").agg(orders=("order_id","size"),late_rate_pct=("late_flag",lambda x:round(x.mean()*100,2))).sort_values("late_rate_pct"))
print("These are associations. They do not prove that support or intervention caused delay; customers may contact support because an order is already going badly, and interventions may target the hardest cases.")

A. Support association


,orders,late_rate_pct,median_delay_min
support_opened,,,
False,500,59.20,2.06
True,1100,55.09,1.16


B. Intervention association


,orders,late_rate_pct
has_intervention,,
False,1170,56.37
True,430,56.44


C. Intervention type


,orders,late_rate_pct
intervention_types,,
PRIORITY_DISPATCH,95,51.16
RESTAURANT_CONTACT,116,52.78
DRIVER_REASSIGNMENT,155,57.82
CUSTOMER_CREDIT,64,66.67


These are associations. They do not prove that support or intervention caused delay; customers may contact support because an order is already going badly, and interventions may target the hardest cases.


## Challenge 6 — KPI linkage and instrumentation

In [7]:
print("PROJECT KPI: reduce late delivery rate")
print("→ outcome: late_delivery_rate, median_delay_among_late")
print("→ workflow/driver metrics: order-to-pickup time, traffic/distance segments, customer friction")
print("→ interventions: intervention rate and type-specific outcomes")
print("→ sources/events: orders, outcomes, customer actions, tickets, interventions, order events, driver events")
print("Controllable by operations: pickup time, intervention routing, restaurant/driver process. Outcomes: late rate and delay severity. Biggest missing event: explicit driver arrival at restaurant. Instrument next: driver_arrived_at_restaurant, consistent event timestamps, and intervention outcome/reason codes.")

PROJECT KPI: reduce late delivery rate
→ outcome: late_delivery_rate, median_delay_among_late
→ workflow/driver metrics: order-to-pickup time, traffic/distance segments, customer friction
→ interventions: intervention rate and type-specific outcomes
→ sources/events: orders, outcomes, customer actions, tickets, interventions, order events, driver events
Controllable by operations: pickup time, intervention routing, restaurant/driver process. Outcomes: late rate and delay severity. Biggest missing event: explicit driver arrival at restaurant. Instrument next: driver_arrived_at_restaurant, consistent event timestamps, and intervention outcome/reason codes.


## Final reflection

The smallest useful model is an order-level journey with carefully aggregated customer interactions, support, interventions, and outcomes. It supports the KPI without pretending that associations are causal.